# Solution 13 — two chats, one database

Each chat id gets its own rows, so two chats can't leak into each other. `ProcessHistory` trims only
what the model sees; `save` appends `new_messages()`, so the file keeps every turn.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
import sqlite3
from pathlib import Path

from pydantic_ai import Agent
from pydantic_ai.capabilities import ProcessHistory
from pydantic_ai.messages import ModelMessagesTypeAdapter, ModelResponse, TextPart, UserPromptPart
from pydantic_ai.models.function import FunctionModel

DB = Path("../data/13s_chats.sqlite")  # its own file, so it never clashes with notebook 13


def save(chat_id, result):
    """Append this run's new messages to one chat."""
    db = sqlite3.connect(DB)
    db.execute("INSERT INTO turns VALUES (?, ?)", (chat_id, result.new_messages_json()))
    db.commit()
    db.close()


def load(chat_id):
    """Rebuild one chat's whole history, oldest turn first."""
    db = sqlite3.connect(DB)
    rows = db.execute(
        "SELECT messages FROM turns WHERE chat_id = ? ORDER BY rowid", (chat_id,)
    ).fetchall()
    db.close()
    history = []
    for row in rows:
        history += ModelMessagesTypeAdapter.validate_json(row[0])
    return history


def prompts_in(messages):
    """Every user prompt in a list of messages, in order."""
    prompts = []
    for message in messages:
        for part in message.parts:
            if isinstance(part, UserPromptPart):
                prompts.append(part.content)
    return prompts


def echo_memory(messages, info):
    """A fake model: it answers with every user prompt it was shown."""
    return ModelResponse(parts=[TextPart(f"I can see: {prompts_in(messages)}")])


def keep_last_3(messages):
    """Keep the last question, its answer, and the new prompt."""
    return messages[-3:]


trimmed = Agent(FunctionModel(echo_memory), capabilities=[ProcessHistory(keep_last_3)])


async def chat(agent, chat_id, prompt):
    """One turn: load the chat, run the agent on it, save the new messages."""
    result = await agent.run(prompt, message_history=load(chat_id))
    save(chat_id, result)
    return result.output

In [ ]:
Path("../data").mkdir(exist_ok=True)
DB.unlink(missing_ok=True)  # a fresh database every time this cell runs
db = sqlite3.connect(DB)
db.execute("CREATE TABLE turns (chat_id TEXT, messages BLOB)")
db.close()

replies = {"alice": [], "bob": []}
for turn in [1, 2, 3]:
    for name in ["alice", "bob"]:  # taking turns: alice, bob, alice, ...
        replies[name].append(await chat(trimmed, name, f"{name}, turn {turn}"))

for name in replies:
    print(name, "last reply:", replies[name][-1])
    print(name, "stored    :", prompts_in(load(name)))

In [ ]:
# 1. No leaks: each chat only ever saw its own prompts.
assert not any("alice" in reply for reply in replies["bob"]), replies["bob"]
assert not any("bob" in reply for reply in replies["alice"]), replies["alice"]

# The model's view really was trimmed: turn 1 is missing from the last reply...
assert "turn 1" not in replies["alice"][-1], replies["alice"][-1]

# 2. ...but the file kept every message of every turn.
for name in replies:
    stored = load(name)
    assert len(stored) == 6, len(stored)
    assert prompts_in(stored) == [f"{name}, turn {t}" for t in [1, 2, 3]], prompts_in(stored)

print("✅ no leaks, and trimming never touched the database")